# 01 · FlashSAC 기초: Bellman target, replay coverage, UTD

목표: SAC의 clipped double-Q target을 계산하고, 과거 policy를 섞은 replay data가 final policy data보다 넓은 범위를 가질 수 있음을 toy example로 확인합니다.

실행: Python 3.10+, NumPy, Matplotlib. 이 notebook은 논문의 로봇 성능을 재현하지 않는 합성 실습입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
np.set_printoptions(precision=4, suppress=True)

## 1. Clipped double-Q Bellman target

두 target critic 중 작은 값을 사용하면 한 critic의 낙관적 과대평가가 target에 그대로 들어가는 것을 줄입니다.

$$y=r+\gamma(\min_j Q_{\bar\phi_j}(s',a')-\alpha\log\pi(a'|s'))$$

In [ ]:
reward = np.array([1.0, 0.2, -0.5, 2.0])
q1_next = np.array([4.0, 1.2, 0.4, 3.0])
q2_next = np.array([3.2, 1.4, 0.1, 2.7])
log_prob = np.array([-0.3, -0.8, -0.1, -0.5])
gamma, alpha = 0.99, 0.2

target = reward + gamma * (np.minimum(q1_next, q2_next) - alpha * log_prob)
optimistic_target = reward + gamma * (np.maximum(q1_next, q2_next) - alpha * log_prob)
print('clipped target   :', target)
print('optimistic target:', optimistic_target)
assert np.all(target <= optimistic_target)

## 2. Replay buffer가 만드는 넓은 coverage

final policy는 한 영역에 집중되어 있다고 가정합니다. Replay buffer에는 학습 중 거쳐 온 여러 behavior policy의 표본이 섞입니다.

In [ ]:
means = np.array([[-2.0, -1.0], [0.0, 1.5], [2.0, -0.5], [0.5, 0.0]])
replay = np.vstack([rng.normal(mean, 0.45, size=(500, 2)) for mean in means])
on_policy = rng.normal([0.5, 0.0], 0.45, size=(2000, 2))

replay_cov_area = np.linalg.det(np.cov(replay.T))
on_policy_cov_area = np.linalg.det(np.cov(on_policy.T))
print(f'covariance determinant - replay: {replay_cov_area:.3f}')
print(f'covariance determinant - final policy: {on_policy_cov_area:.3f}')
assert replay_cov_area > on_policy_cov_area

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
axes[0].scatter(replay[:, 0], replay[:, 1], s=5, alpha=0.2)
axes[0].set_title('Toy off-policy replay')
axes[1].scatter(on_policy[:, 0], on_policy[:, 1], s=5, alpha=0.2, color='tab:orange')
axes[1].set_title('Toy final-policy rollout')
for ax in axes:
    ax.set_xlabel('state feature')
    ax.set_ylabel('action feature')
plt.tight_layout();

## 3. UTD 비율

논문의 GPU 설정은 새 transition 1024개당 gradient update 2회입니다. UTD를 낮추면 update 비용은 줄지만, 큰 batch·모델·안정화 구조가 없으면 underfitting이 될 수 있습니다.

In [ ]:
new_transitions = 50_000_000
updates = new_transitions * 2 / 1024
samples_seen = updates * 2048
print(f'gradient updates: {updates:,.0f}')
print(f'batch samples consumed (with reuse): {samples_seen:,.0f}')
print(f'effective sample reuses per new transition: {samples_seen/new_transitions:.2f}')
assert samples_seen / new_transitions == 4.0

## 정리

- clipped double-Q는 낙관적 target을 줄입니다.
- replay의 넓은 coverage는 장점이지만 오래되거나 품질 낮은 데이터도 포함할 수 있습니다.
- FlashSAC의 낮은 UTD는 각 update의 큰 batch와 높은 데이터 처리량을 전제로 해석해야 합니다.

다음 notebook에서는 entropy target, reward scaling과 noise repetition을 구현합니다.